In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Dropout, RandomFlip, RandomRotation
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger


2026-04-02 00:41:36.312751: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775090496.527113      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775090496.592610      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775090497.081651      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775090497.081719      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775090497.081722      24 computation_placer.cc:177] computation placer alr

In [2]:
# --- 1. KAGGLE CONFIGURATION ---
DATASET_DIR = "/kaggle/input/datasets/sakibmansuri/food101-custom-dataset/dataset"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "test")

# Kaggle requires all generated files (models, text files) to be saved in the /kaggle/working/ directory
WORKING_DIR = "/kaggle/working"

IMG_SIZE = (224, 224)
BATCH_SIZE = 64 # Kaggle GPUs have more VRAM, so we can double the batch size for faster training!

print(f"📂 Scanning directories at {DATASET_DIR}...")


📂 Scanning directories at /kaggle/input/datasets/sakibmansuri/food101-custom-dataset/dataset...


In [3]:
# --- 2. DATA LOADING & CLASS DETECTION ---
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

# Extract class names dynamically based on your custom folders
class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"✅ Detected {num_classes} custom categories.")

Found 75750 files belonging to 101 classes.


I0000 00:00:1775090576.535464      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 25250 files belonging to 101 classes.
✅ Detected 101 custom categories.


In [4]:
# --- 3. AUTO-GENERATE classes.txt ---
classes_file_path = os.path.join(WORKING_DIR, "classes.txt")
with open(classes_file_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(class_names))
print(f"📝 Auto-generated '{classes_file_path}' for your backend.")

📝 Auto-generated '/kaggle/working/classes.txt' for your backend.


In [5]:
# --- 4. PREPROCESSING ---
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

train_dataset = train_dataset.map(lambda x, y: (preprocess_input(x), y))
val_dataset = val_dataset.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)


In [6]:
# --- 5. MODEL ARCHITECTURE(Upgraded for Fine-Tuning) ---
print("🏗️ Building MobileNetV2 Transfer Learning model...")

base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

# 1. Unfreeze the base model
base_model.trainable = True 


# 2. Re-freeze the bottom layers (Keep foundational knowledge like edges and curves)
# MobileNetV2 has 154 layers. We will only let it re-train the top 54 layers for food!
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

inputs = Input(shape=(224, 224, 3))
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.2)(x)

# Note: We keep training=False here so BatchNorm layers don't get messed up during fine-tuning
x = base_model(x, training=False) 
x = GlobalAveragePooling2D()(x)

# Add a dense hidden layer to process the newly unfreezed features better
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x) 

outputs = Dense(num_classes, activation='softmax')(x)
model = Model(inputs, outputs)


🏗️ Building MobileNetV2 Transfer Learning model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
# --- 6. COMPILE & TRAIN ---

# 3. Compile with a MUCH LOWER learning rate
# Since we are tweaking the base model, we want baby steps so we don't erase its memory.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), # Reduced from 0.001 to 0.0001
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

# Save the best model to Kaggle's working directory
model_save_path = os.path.join(WORKING_DIR, "model.h5")
checkpoint = ModelCheckpoint(model_save_path, save_best_only=True, monitor='val_accuracy')

# Increased patience to 5 so it doesn't quit too early during fine-tuning
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

log_csv_path = os.path.join(WORKING_DIR, "training_logs.csv")
csv_logger = CSVLogger(log_csv_path, append=False)

print("🚀 Starting GPU training process...")
# changes epochs from 10 -> 20.
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20, 
    callbacks=[checkpoint, early_stop, csv_logger]
)

print(f"\n🎉 Training complete!")
print(f"✅ Model saved to: {model_save_path}")
print(f"✅ Classes saved to: {classes_file_path}")
print("👉 Look at the 'Output' tab on the right menu in Kaggle to download these files!")

🚀 Starting GPU training process...
Epoch 1/20


I0000 00:00:1775090612.955871      73 cuda_dnn.cc:529] Loaded cuDNN version 91002


1184/1184 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.2044 - loss: 3.6159

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 220s 171ms/step - accuracy: 0.2045 - loss: 3.6152 - val_accuracy: 0.4872 - val_loss: 2.0517
Epoch 2/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.5292 - loss: 1.9096

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 162s 136ms/step - accuracy: 0.5292 - loss: 1.9095 - val_accuracy: 0.6120 - val_loss: 1.5022
Epoch 3/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.6034 - loss: 1.5856

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 162s 137ms/step - accuracy: 0.6034 - loss: 1.5855 - val_accuracy: 0.6697 - val_loss: 1.2493
Epoch 4/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.6484 - loss: 1.3871

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 161s 136ms/step - accuracy: 0.6484 - loss: 1.3871 - val_accuracy: 0.7092 - val_loss: 1.0906
Epoch 5/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.6764 - loss: 1.2566

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 161s 136ms/step - accuracy: 0.6764 - loss: 1.2566 - val_accuracy: 0.7318 - val_loss: 1.0017
Epoch 6/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 160s 135ms/step - accuracy: 0.6992 - loss: 1.1542 - val_accuracy: 0.7255 - val_loss: 1.0181
Epoch 7/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.7193 - loss: 1.0686

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 158s 133ms/step - accuracy: 0.7193 - loss: 1.0686 - val_accuracy: 0.7445 - val_loss: 0.9573
Epoch 8/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.7371 - loss: 0.9952

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 157s 132ms/step - accuracy: 0.7371 - loss: 0.9951 - val_accuracy: 0.7596 - val_loss: 0.8836
Epoch 9/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 156s 132ms/step - accuracy: 0.7497 - loss: 0.9324 - val_accuracy: 0.7493 - val_loss: 0.9316
Epoch 10/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 156s 132ms/step - accuracy: 0.7683 - loss: 0.8683 - val_accuracy: 0.7526 - val_loss: 0.9426
Epoch 11/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 156s 132ms/step - accuracy: 0.7777 - loss: 0.8265 - val_accuracy: 0.7531 - val_loss: 0.9554
Epoch 12/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 158s 133ms/step - accuracy: 0.7903 - loss: 0.7730 - val_accuracy: 0.7574 - val_loss: 0.9516
Epoch 13/20
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 158s 133ms/step - accuracy: 0.7971 - loss: 0.7356 - val_accuracy: 0.7585 - val_loss: 0.9246

🎉 Training complete!
✅ Model saved to: /kaggle/working/model.h5
✅ Classes saved to: /kaggle/working/classes.txt
👉 Look at the 'Output' tab on the right menu in Kaggle to download these files!


In [8]:
# --- 7. EVALUATION & VISUALIZATION ---
print("\n📊 Generating Evaluation Reports...")

# A. Plot Training History (Accuracy & Loss curves)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy over Epochs')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss over Epochs')

history_path = os.path.join(WORKING_DIR, "training_history.png")
plt.savefig(history_path)
plt.close()

# B. Get True Labels and Predictions
print("🔍 Analyzing model mistakes (this takes a moment)...")
y_true = np.concatenate([y for x, y in val_dataset], axis=0)
y_true_classes = np.argmax(y_true, axis=1)

y_pred = model.predict(val_dataset)
y_pred_classes = np.argmax(y_pred, axis=1)

# C. Classification Report (Precision, Recall, F1-Score)
report = classification_report(y_true_classes, y_pred_classes, target_names=class_names)
report_path = os.path.join(WORKING_DIR, "classification_report.txt")
with open(report_path, 'w') as f:
    f.write(report)

# D. Confusion Matrix (Visual Heatmap)
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(24, 20))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual Food')
plt.xlabel('Predicted Food')
cm_path = os.path.join(WORKING_DIR, "confusion_matrix.png")
plt.savefig(cm_path)
plt.close()

print("\n🎉 ALL DONE! Your outputs have been saved to the working directory.")


📊 Generating Evaluation Reports...
🔍 Analyzing model mistakes (this takes a moment)...
395/395 ━━━━━━━━━━━━━━━━━━━━ 29s 71ms/step

🎉 ALL DONE! Your outputs have been saved to the working directory.
